In [1]:
import numpy as np
from util.utility import get_mongo_collection
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import OllamaLLM
from langchain.chains import RetrievalQA

In [2]:
def generate_text(game_obj):
    text_obj = dict()
    str_obj = ''
    if game_obj.get('steam-description', '#') != '#':
        str_obj += 'Description: ' + game_obj['steam-description'] + '\n'
        text_obj['Intro'] = game_obj['steam-description']
    if game_obj.get('steam-summary', '#') != '#':
        str_obj += 'Summary: ' + game_obj['steam-summary'] + '\n'
    if str_obj not in ['', '#']:
        text_obj['steam-text'] = str_obj
    
    if game_obj.get('metacritics-description', '#') != '#':
        game_obj['metacritics-text'] = game_obj['metacritics-description']  + '\n'
        if 'Intro' not in text_obj:
            text_obj['Intro'] = game_obj['metacritics-text']

    if game_obj.get('rawg-description', '#') != '#':
        game_obj['rawg-text'] = game_obj['rawg-description'].replace('<p>', '').replace('</p>', '')  + '\n'
    
    str_obj = ''
    if game_obj.get('wikipedia-summary', '#') != '#':
        str_obj += 'Summary: ' + game_obj['wikipedia-summary'] + '\n'
        if 'Intro' not in text_obj:
            text_obj['Intro'] = game_obj['wikipedia-summary']
    if game_obj.get('wikipedia-gameplay', '#') != '#':
        str_obj += 'Gameplay: ' + game_obj['wikipedia-gameplay'] + '\n'
    if game_obj.get('wikipedia-plot', '#') != '#':
        str_obj += 'Plot: ' + game_obj['wikipedia-plot'] + '\n'
    if game_obj.get('wikipedia-synopsis', '#') != '#':
        str_obj += 'Synopsis: ' + game_obj['wikipedia-synopsis'] + '\n'
    if str_obj not in ['', '#']:
        text_obj['wikipedia-text'] = str_obj

    str_obj = ''
    if game_obj.get('giantbomb-intro', '#') != '#':
        str_obj += 'Intro: ' + game_obj['giantbomb-intro'] + '\n'
        if 'Intro' not in text_obj:
            text_obj['Intro'] = game_obj['giantbomb-intro']
    if game_obj.get('giantbomb-description', '#') != '#':
        str_obj += 'Description: ' + game_obj['giantbomb-description'] + '\n'
    if str_obj not in ['', '#']:
        text_obj['giantbomb-text'] = str_obj

    str_obj = ''
    if game_obj.get('igdb-summary', '#') != '#':
        str_obj += 'Summary: ' + game_obj['igdb-summary'] + '\n'
    if game_obj.get('igdb-story', '#') != '#':
        str_obj += 'Story: ' + game_obj['igdb-story'] + '\n'
    if str_obj not in ['', '#']:
        text_obj['igdb-text'] = str_obj

    if game_obj.get('backloggd-description', '#') != '#':
        text_obj['backloggd-text'] = game_obj['backloggd-description']  + '\n'

    if game_obj.get('moby-description', '#') != '#':
        text_obj['moby-text'] = game_obj['moby-description']  + '\n'

    if game_obj.get('gamesdb-description', '#') != '#':
        text_obj['gamesdb-text'] = game_obj['gamesdb-description']  + '\n'
    
    return text_obj

In [ ]:
games_text = []
texts = []
metadatas = []
for game in list(get_mongo_collection().find({}).sort({'_id': -1}).limit(270)):
    title = game.get('title', '#')
    games_text =  generate_text(game)

    for source, text in games_text.items():
        texts.append(text)
        metadatas.append({
            "game": title,
            "source": source
        })

    for key in ['giantbomb-franchises', 'giantbomb-platforms', 'giantbomb-themes', 'igdb-game_engines', 'igdb-keywords',
                'igdb-player_perspectives', 'igdb-themes', 'moby-aliases', 'moby-genres', 'moby-reviews',
                'moby-tags', 'steam-tags', 'wikipedia-genre', 'giantbomb-developers', 'giantbomb-genres', 'giantbomb-publishers',
                'igdb-genres', 'steam-developers', 'steam-genres', 'franchise']:
        value = game.get(key, None)
        if isinstance(value, str):
            # simple key-value text
            texts.append(value)
            metadatas.append({"source": key, "game": title})

        elif value and key == 'moby-genres':
            moby_genres = ''
            for subkey, subvalue in value.items():
                moby_genres += subkey + ': ' + subvalue + ' / '
            texts.append(moby_genres)
            metadatas.append({"source": "moby-genres", "game": title})

        elif isinstance(value, dict):
            # nested dict like moby-genres or moby-reviews
            for subkey, subvalue in value.items():
                if isinstance(subvalue, str):
                    texts.append(subvalue)
                    metadatas.append({"source": f"{key}.{subkey}", "game": title})
                elif isinstance(subvalue, list):
                    for item in subvalue:
                        # handle list of dicts (like reviews)
                        text_piece = item.get("review") or str(item)
                        texts.append(text_piece)
                        # keep numeric values like score in metadata
                        md = {"source": f"{key}.{subkey}", "game": title}
                        metadatas.append(md)

In [ ]:
# Load embeddings (local model)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Create Chroma vector database
db = Chroma.from_texts(
    texts,
    embedding=embeddings,
    metadatas=metadatas,
    persist_directory="./chroma_vg_db" 
)

In [3]:
# Loading embeddings (local model) and Chroma DB
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = Chroma(
    persist_directory="./chroma_vg_db",
    embedding_function=embeddings
)

In [56]:
# Get game
game = db.get(where={'game': 'Absolum'})
game

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

In [24]:
# Add new games to the DB as needed
BATCH_SIZE = 500

for i in range(0, len(texts), BATCH_SIZE):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_metadatas = metadatas[i:i + BATCH_SIZE]

    db.add_texts(
        texts=batch_texts,
        metadatas=batch_metadatas,
    )

In [3]:
local_llm = OllamaLLM(model="mistral")

In [23]:
source_list = ['steam-text', 'igdb-text', 'wikipedia-text', 'giantbomb-text', 'gamesdb-text',
                             'rawg-text', 'mobygames-text', 'backloggd-text', 'metacritic-text']

games = ['The Last of Us']
filters = {
    "$and": [
        {"source": {"$in": source_list}},
        {"game": {"$in": games}}
    ]
}

retriever = db.as_retriever(search_kwargs={"k": 20, "filter": filters})

qa = RetrievalQA.from_chain_type(
    llm=local_llm,
    retriever=retriever,
    return_source_documents=True
)

query = "what is the ending of The Last of Us in details?"
result = qa.invoke(query)
print(result["result"])

 In "The Last of Us," the ending is quite emotional and open-ended, allowing for interpretation.

Spoilers ahead!

After a long journey, Joel and Ellie finally reach the hospital where the Fireflies are located. The Fireflies, a militia group seeking a cure for the fungal infection that has plagued humanity, want to extract Ellie's immune brain tissue to create a cure. Ellie, however, is unwilling to undergo the operation as she forms a strong bond with Joel during their journey.

Joel deceives the Fireflies by pretending that Ellie underwent the operation, but secretly escapes with her instead. They travel through a dangerous quarantine zone filled with infected and hostile survivors to reach a safer location.

Upon reaching the boat that will take them away, Joel is confronted by Tess, a friend who helped him in his smuggling operations earlier in the game. Tess had been gravely wounded during their escape from the Fireflies, and she asks Joel to fulfill a promise he made her: to kee

In [23]:
docs = db._collection.get(
    where={
        "$and": [
            {"game": "Hollow Knight"},
            {"source": {"$in": source_list}}
        ]
    },
    include=["embeddings", "metadatas", "documents"]
)

In [23]:
docs = db._collection.get(
        where={
            '$and': [
                {'game': 'Unreal Tournament 2004'},
                {'source': 'giantbomb-franchises'}
            ]
        },
        include=['documents']
    )

In [61]:
def get_similar_games(game_name: str, db, top_k=5, reject_same_franchise=True):
    '''Compute average embedding for one game and find similar ones.'''
    # fetch all docs for the game
    try:
        # source_list = ['steam-text', 'igdb-text', 'wikipedia-text', 'giantbomb-text', 'gamesdb-text',
        #                         'rawg-text', 'mobygames-text', 'backloggd-text', 'metacritic-text']
        source_list = ['steam-text', 'wikipedia-text']
        source_filters = {'source': {'$in': source_list}} 
        docs = db._collection.get(
            where={
                '$and': [
                    {'game': game_name},
                    {'source': {'$in': source_list}}
                ]
            },
            include=['embeddings']
        )

        doc_franchise = db._collection.get(
            where={
                '$and': [
                    {'game': game_name},
                    {'source': {'$in': ['giantbomb-franchises', 'franchise']}}
                ]
            },
            include=['documents']
        )

        franchises = set()
        if len(doc_franchise['documents']) > 0:
            franchises = set(doc_franchise['documents'][0].split('; '))
            if len(doc_franchise['documents']) > 1:
                franchises.update(set(doc_franchise['documents'][1].split('; ')))

        # compute centroid embedding for this game
        avg_vector = np.mean(docs['embeddings'], axis=0)

        # query by vector directly
        results = db.similarity_search_by_vector(avg_vector, k=top_k, filter=source_filters)
        
        # Group by game, exclude self and same-franchise games
        similar = []
        seen = {game_name.lower()}
        for doc in results:
            g = doc.metadata.get('game', '').strip()
            if not g or g.lower() in seen:
                continue
            
            if reject_same_franchise:
                # Get this result's franchises
                doc_franchise = db._collection.get(
                    where={
                        '$and': [
                            {'game': g},
                            {'source': {'$in': ['giantbomb-franchises', 'franchise']}}
                        ]
                    },
                    include=['documents']
                )

                if len(doc_franchise['documents']) > 0:
                    doc_franchises = set(doc_franchise['documents'][0].split('; '))
                    if len(doc_franchise['documents']) > 1:
                        doc_franchises.update(set(doc_franchise['documents'][1].split('; ')))
                else:
                    doc_franchises = set()

                # Skip if shares any franchise with the target
                if franchises and (franchises & doc_franchises):
                    continue

            similar.append(g)
            seen.add(g.lower())
            if len(similar) >= top_k:
                break
    except Exception as e:
        return ['Error']

    return similar

In [ ]:
similar_dict = dict()
for game in get_mongo_collection().distinct('title'):
    similar = get_similar_games(game, db, top_k=100)[:10]
    if len(similar) > 0:
        print(game, '#', similar)
        if similar[0] != 'Error':
            similar_dict[game] = similar

In [64]:
import json
with open('similar_games.json', 'w', encoding='utf-8') as file:
    json.dump(similar_dict, file)

In [ ]:
used_titles = set()
for sims in similar_dict.values():
    used_titles.update(sims)

all_titles = set(get_mongo_collection().distinct('title'))
unused_titles = all_titles - used_titles

In [67]:
unused_titles

{'#BLUD',
 '10,000 Bullets',
 "8Doors: Arum's Afterlife Adventure",
 'A Tower Full of Cats',
 'AWAKEN - Astral Blade',
 'Absolum',
 'Adore',
 'Aidyn Chronicles - The First Mage',
 'Aladdin',
 'Alfred Chicken',
 'And Roger',
 'Anomaly Agent',
 'Ape Escape: On the Loose',
 'Apollo Justice - Ace Attorney',
 'Apollo Justice: Ace Attorney Trilogy',
 "Archer Maclean's Mercury",
 'Arkanoid - Doh It Again',
 'Armored Core - Formula Front',
 'Armored Core - Project Phantasma',
 'Armored Core V',
 "Assassin's Creed - Revelations",
 "Assassin's Creed - The Ezio Collection",
 'Assemble with Care',
 "Astro's Playroom",
 "Asura's Wrath",
 'Atomic Runner',
 'Aurion Legacy of the Kori-Odan',
 'Aviano',
 'BALL X PIT',
 'Ballex 2 - The Hanging Gardens',
 'Bangai-O Spirits',
 'Batman',
 'Batman Vengeance',
 'Battle Mania',
 'Bayonetta Origins Cereza and the Lost Demon',
 'Beacon',
 'Beholder Conductor',
 'Biped',
 'Bishi Bashi Special',
 'Black Mesa',
 'Blackhawk',
 'Bloody Roar 4',
 'Bloody Spell',
 'Bl

In [15]:
similar_games_ai = dict()
all_games = list(get_mongo_collection().find({}, {'title': 1, '_id': 0}))
for game in all_games:
    try:
        similar = get_similar_games(game['title'], db, top_k=10)
        similar_games_ai[game['title']] = similar
        print(game['title'], '#', similar[0])
    except Exception as e:
        print('Error', game['title'], e)

import json
with open('similar_games_ai.json', 'w', encoding='utf-8') as f:
    json.dump(similar_games_ai, f, indent=4)